# Model service: the model as a callable building block
**Internet of Production – AI, week 3, block 5** · about 30 min · work in pairs

**Thread:** A model in a notebook helps nobody on the shop floor. The monitor, the agents of the next block and the maintenance dashboard need it as a **service**: they send the last minutes of energy data and get back a score and an alarm. This notebook starts such a service, calls it, tries to break it, and rolls out a new model version without touching the code.

**Architecture**
```
monitor / agent  --HTTP POST /predict-->  blockage_service.py  --loads by alias-->  MLflow registry
   (360 values)    <--score, alarm, version--   (Flask)                            blockage-zrule@production
```
**Interface contract:** input = JSON with `energy` (at least 360 values, Wh per **printing** minute, oldest first) and optional `model` (`zrule` or `knn`); output = `score`, `alarm`, `version`, `features`, `ms`. Errors are answered with HTTP 400 and a message, never with a made-up score.

*Requirement:* notebook 02 must have been run (the registered models must exist).

In [ ]:
import sys, subprocess, time, json, logging, warnings
sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, requests
from pathlib import Path
import mlflow, mlflow.pyfunc
logging.getLogger('mlflow').setLevel(logging.ERROR); warnings.filterwarnings('ignore')     # keep the output readable
from blockage_core import *

DATA_NEW = Path('../../00_Data/printfarm_2026')
STORE = Path('..').resolve()
TRACKING_URI = f"sqlite:///{(STORE / 'mlflow.db').as_posix()}"
mlflow.set_tracking_uri(TRACKING_URI)
PORT = 5005
URL = f'http://127.0.0.1:{PORT}'

printers = {k: make_blockage_data(DATA_NEW, k) for k in range(2)}      # printers 0 and 1 are enough here
minute_data, _ = load_printer(DATA_NEW, 0)
printing = minute_data[minute_data['State'] == 'printing']['E']        # Wh per printing minute of printer 0

def window_at(t, n=WINDOW):
    """The last n printing minutes up to time t as a plain list: this is what the monitor will send."""
    return printing.loc[:t].iloc[-n:].tolist()

def call(energy=None, model='zrule', raw=None):
    """POST to the service; returns (HTTP status, answer)."""
    body = raw if raw is not None else {'energy': energy, 'model': model}
    r = requests.post(f'{URL}/predict', data=json.dumps(body), headers={'Content-Type': 'application/json'}, timeout=10)   # json.dumps also lets NaN through, as a careless sender would
    return r.status_code, r.json()

## Step 1: Start the service and check its health
The service runs as a separate process (`blockage_service.py`, in the folder above). It loads the models with the alias `production` from the registry.

In [ ]:
service = subprocess.Popen([sys.executable, 'blockage_service.py', '--port', str(PORT), '--tracking-uri', TRACKING_URI],
                           cwd='..', stdout=open('../service.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        health = requests.get(f'{URL}/health', timeout=1).json(); break
    except requests.ConnectionError:
        time.sleep(1)
print(json.dumps(health, indent=1))

## Step 2: First calls with real data
`window_at(t)` returns the last 360 printing minutes up to time `t`. Take one time stamp shortly before a blockage (label `y = 1`) and one in normal operation (`y = 0`) from printer 0 and ask the service.

In [ ]:
d0, blk0 = printers[0]
t_warn = d0[d0['y'] == 1].index[len(d0[d0['y'] == 1]) // 2]        # a time with a blockage ahead
t_ok = d0[d0['y'] == 0].index[len(d0[d0['y'] == 0]) // 2]          # a time in normal operation

for label, t in [('warning', t_warn), ('normal', t_ok)]:
    for model in ['zrule', '___']:                                  # the second model of the registry
        status, ans = call(window_at(t), model)
        print(f"{label:8s} {t}  {model:5s} HTTP {status}  score {ans['score']:6.2f}  alarm {ans['alarm']}  version {ans['version']}  {ans['ms']} ms")

**Do both models answer correctly? Why is the `score` of the two models not comparable, and what does it mean for a dashboard that shows both?**

*Your answer:*

## Step 3: Try to break it
A service that only works with perfect input is not a service. Predict for each case below what the service answers (HTTP status), then run it.

In [ ]:
good = window_at(t_ok)
cases = {
    'too short (10 values)':      {'energy': good[-10:]},
    'NaN in the window':          {'energy': good[:-1] + [float('nan')]},
    'wrong unit (mWh)':           {'energy': [v * 1000 for v in good]},
    'text instead of numbers':    {'energy': ['a'] * 400},
    'key missing':                {'values': good},
    'unknown model':              {'energy': good, 'model': 'lstm'},
    'longer window (2000 values)': {'energy': window_at(t_ok, 2000)},
}
for name, body in cases.items():
    status, ans = call(raw=body)
    print(f"{name:30s} HTTP {status}  {ans.get('error', 'score ' + str(round(ans.get('score', float('nan')), 2)))}")

**Which cases does the service reject, which does it accept, and is that right? Which protection is still missing before the service may run unattended for months?**

*Your answer:*

## Step 4: Is the service consistent with the offline pipeline?
The features are computed twice: in the data pipeline (rolling windows over the whole series) and in the service (from a window of 360 values). If they differ, the model sees other numbers in operation than in training. Compare both for 50 time stamps of printer 0.

In [ ]:
sample = d0.sample(50, random_state=1)
diffs, latency = [], []
for t, row in sample.iterrows():
    status, ans = call(window_at(t), 'zrule')
    diffs.append([abs(ans['features'][f] - row[f]) for f in FEATURES]); latency.append(ans['ms'])
print('largest difference per feature (offline vs service):', dict(zip(FEATURES, np.max(diffs, axis=0).round(12).tolist())))
print(f'service time per call: median {np.median(latency):.1f} ms, 95th percentile {np.___(latency, .95):.1f} ms')

**Are the features identical? Is the response time sufficient for the monitor that asks six printers once per minute?**

*Your answer:*

## Step 5: Roll out a new version and roll back
Register a **second version** of the z-value rule with a different limit, move the alias `production` to it, and tell the service to reload. No code and no restart. Then roll back.

In [ ]:
client = mlflow.MlflowClient()
v1 = client.get_model_version_by_alias('blockage-zrule', 'production')
base = mlflow.pyfunc.load_model(f'models:/blockage-zrule@production').unwrap_python_model()

def score_of(t, model='zrule'):
    return call(window_at(t), model)[1]

print('before:', {k: score_of(t_warn)[k] for k in ('version', 'score', 'alarm')})

NEW_LIMIT = ___                                                 # a clearly stricter limit than the default, e.g. 3.5
with mlflow.start_run(run_name='stricter limit'):
    info = mlflow.pyfunc.log_model(name='model', python_model=ZRuleModel(base.mean, base.sd, NEW_LIMIT), input_example=d0[FEATURES].head(2).astype(float),
                                   pip_requirements=['mlflow', 'scikit-learn', 'pandas', 'numpy'])
v2 = mlflow.register_model(info.model_uri, 'blockage-zrule')
client.set_registered_model_alias('blockage-zrule', 'production', v2.version)
requests.post(f'{URL}/reload')
print('after: ', {k: score_of(t_warn)[k] for k in ('version', 'score', 'alarm')})

client.set_registered_model_alias('blockage-zrule', 'production', v1.version)          # roll back
requests.post(f'{URL}/reload')
print('rolled back:', {k: score_of(t_warn)[k] for k in ('version', 'score', 'alarm')})

**What changed for the caller when the version changed, and what did not? Who should be allowed to move the alias `production`, and what would you demand before it is moved?**

*Your answer:*

Stop the service when you are done. **The monitor of block 6 starts its own copy, so keep port 5005 free.**

In [ ]:
service.terminate(); service.wait(); print('service stopped')